In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from itertools import combinations
import matplotlib.pyplot as plt

In [ ]:
def fetch_log_prices(tickers: list, start_date: str, end_date: str) -> pd.DataFrame:
    """
    Downloads Adjusted Close prices for multiple tickers, handles missing data,
    and computes their log-prices for pairs trading analysis.
    """
    print(f"Downloading data for {len(tickers)} tickers from {start_date} to {end_date}...")
    
    try:
        # Download data silently
        data = yf.download(tickers, start=start_date, end=end_date, progress=False, auto_adjust=True)['Close']
        data = np.log(data)  # Compute log-prices
        
        # Error handling for empty data or missing tickers
        if data.empty:
            raise ValueError("No data returned from yfinance. Check date ranges and network connection.")
        for ticker in tickers:
            if ticker not in data.columns:
                raise ValueError(f"Ticker {ticker} is invalid or has no data.")
            
        # Drop NaN values to ensure aligned time series for regression
        data = data.dropna()
        
        if len(data) < 30:
            raise ValueError("Insufficient data points after dropping NaNs (need at least 30 days of overlap).")

        return data
        
    except Exception as e:
        print(f"Error fetching data: {str(e)}")
        return pd.DataFrame()

In [ ]:
def calculate_npd(log_prices: pd.DataFrame, ticker1: str, ticker2: str) -> float:
    """
    Computes the Normalized Price Distance (NPD) using log-prices.
    Formula: Mean of squared differences of shifted log-prices (y_t - y_0)[cite: 74, 75].
    """
    y1 = log_prices[ticker1]
    y2 = log_prices[ticker2]
    
    # Normalize by subtracting the initial value (y_0) 
    y1_norm = y1 - y1.iloc[0]
    y2_norm = y2 - y2.iloc[0]
    
    # Calculate average squared distance
    npd = np.mean((y1_norm - y2_norm)**2)
    return npd

def test_cointegration(log_data: pd.DataFrame, ticker1: str, ticker2: str):
    """
    Performs OLS to find the hedge ratio and runs ADF on the spread[cite: 87, 97].
    Returns: (p-value, gamma, spread)
    """
    try:
        y1 = log_data[ticker1]
        y2 = log_data[ticker2]
        
        # OLS regression to determine gamma 
        X = sm.add_constant(y2)
        model = sm.OLS(y1, X).fit()
        gamma = model.params.iloc[1]
        
        # Construct the spread z_t = y1 - gamma * y2 [cite: 83]
        spread = y1 - gamma * y2
        
        # ADF Test for stationarity [cite: 91, 97]
        adf_result = adfuller(spread, autolag='AIC')
        p_value = adf_result[1]
        
        return p_value, gamma, spread
    except Exception:
        return 1.0, None, None

def discover_best_pair(tickers: list, start_date: str, end_date: str, top_k: int = 5):
    """
    Stage 1: Prescreening using NPD[cite: 73].
    Stage 2: Statistical testing using ADF on the Top-K candidates[cite: 77].
    """

    log_prices = fetch_log_prices(tickers, start_date, end_date)
    
    if log_prices.empty or len(log_prices.columns) < 2:
        print("Insufficient data for the provided tickers.")
        return None

    log_prices.columns = [f"{t}_log" for t in log_prices.columns]
    
    # --- STAGE 1: PRESCREENING (NPD) ---
    all_pairs = list(combinations(log_prices.columns, 2))
    screening_results = []
    
    for t1, t2 in all_pairs:
        npd_score = calculate_npd(log_prices, t1, t2)
        screening_results.append({'pair': (t1, t2), 'npd': npd_score})
    
    # Rank by lowest distance (heuristic for cointegration) [cite: 73, 74]
    screened_pairs = sorted(screening_results, key=lambda x: x['npd'])[:top_k]
    
    print(f"Prescreening complete. Testing top {top_k} candidates for cointegration...")
    
    # --- STAGE 2: COINTEGRATION TESTING (ADF) ---
    final_results = []
    for candidate in screened_pairs:
        t1, t2 = candidate['pair']
        p_val, gamma, spread = test_cointegration(log_prices, t1, t2)
        
        final_results.append({
            'tickers': (t1, t2),
            'npd': candidate['npd'],
            'p_value': p_val,
            'gamma': gamma
        })
        print(f"Tested {t1}-{t2}: NPD={candidate['npd']:.4f}, ADF p-value={p_val:.4f}")

    # Pick the best (lowest p-value) [cite: 95, 118]
    best_pair = min(final_results, key=lambda x: x['p_value'])
    
    print("\n" + "="*50)
    print("PAIR DISCOVERY SUMMARY")
    print("="*50)
    if best_pair['p_value'] < 0.05:
        print(f"BEST PAIR: {best_pair['tickers'][0]} & {best_pair['tickers'][1]}")
        print(f"ADF p-value: {best_pair['p_value']:.5f}")
        print(f"Hedge Ratio (Gamma): {best_pair['gamma']:.4f}")
        print(f"NPD (Prescreen): {best_pair['npd']:.6f}")
    else:
        print("No statistically significant cointegrated pairs found (p > 0.05).")
    print("="*50)
    
    return best_pair

if __name__ == "__main__":
    # Example execution using large-cap technology stocks
    TICKER_LIST = ["AAPL", "MSFT", "GOOGL", "META", "AMZN", "NVDA"]
    START = "2022-01-01"
    END = "2024-01-01"
    
    best = discover_best_pair(TICKER_LIST, START, END)

In [ ]:
def analyze_market_pair(ticker_y, ticker_x, start_date, end_date, title):
    # 1. Download and Prepare Log-Prices
    data = fetch_log_prices([ticker_y, ticker_x], start_date, end_date)
    
    y = data[ticker_y]
    x = data[ticker_x]
    
    # 2. OLS Regression to find the Spread (Residual)
    X = sm.add_constant(x)
    model = sm.OLS(y, X).fit()
    gamma = model.params.iloc[1]
    intercept = model.params.iloc[0]
    residual = y - (gamma * x + intercept)
    
    # 3. ADF Statistical Test
    adf_result = adfuller(residual, autolag='AIC')
    p_value = adf_result[1]
    
    # 4. Results Summary
    print(f"--- Analysis for {ticker_y} vs {ticker_x} ({start_date[:4]}-{end_date[:4]}) ---")
    print(f"Hedge Ratio (gamma): {gamma:.4f}")
    print(f"ADF p-value:         {p_value:.4f}")
    status = "COINTEGRATED" if p_value < 0.05 else "NOT COINTEGRATED"
    print(f"Conclusion:          {status}\n")
    
    return residual, p_value

# Execution and Plotting
plt.figure(figsize=(12, 8))

# Case 1: EWC - EWA (Mimic Figure 15.10)
# Estimation period: 2016-2019 
res_ewa, p_ewa = analyze_market_pair("EWC", "EWA", "2016-01-01", "2019-12-31", "EWC-EWA")
plt.subplot(2, 1, 1)
res_ewa.plot(color='blue', title=f"Figure 15.10: Cointegration residual for EWC-EWA (p={p_ewa:.4f})")
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.ylabel("Residual")

# Case 2: KO - PEP (Mimic Figure 15.11)
# Estimation period: 2017-2019 
res_ko, p_ko = analyze_market_pair("PEP", "KO", "2017-01-01", "2019-12-31", "PEP-KO")
plt.subplot(2, 1, 2)
res_ko.plot(color='red', title=f"Figure 15.11: Cointegration residual for PEP-KO (p={p_ko:.4f})")
plt.axhline(0, color='black', linestyle='--', alpha=0.5)
plt.ylabel("Residual")

plt.tight_layout()
plt.show()

In [ ]:
def backtest_strategy(log_prices, gamma, intercept, threshold, window=126):
    # Calculate Spread: z = y - (gamma * x + intercept)
    # Assuming EWA is y and EWC is x
    spread = log_prices['EWA'] - (gamma * log_prices['EWC'] + intercept)
    
    # Rolling Z-Score (6 months)
    rolling_mean = spread.rolling(window=window).mean()
    rolling_std = spread.rolling(window=window).std()
    z_score = (spread - rolling_mean) / rolling_std
    
    # Strategy Logic
    positions = pd.Series(0, index=log_prices.index)
    current_pos = 0
    
    for i in range(1, len(z_score)):
        z = z_score.iloc[i]
        if np.isnan(z): continue
        
        if current_pos == 0:
            if z > threshold:
                current_pos = -1 # Short Spread
            elif z < -threshold:
                current_pos = 1  # Long Spread
        elif current_pos == 1:
            if z >= 0: # Exit Long
                current_pos = 0
        elif current_pos == -1:
            if z <= 0: # Exit Short
                current_pos = 0
        
        positions.iloc[i] = current_pos

    # Calculate Returns
    # Spread return = d(y) - gamma * d(x)
    log_returns = log_prices.diff()
    spread_return = log_returns['EWA'] - gamma * log_returns['EWC']
    strategy_returns = positions.shift(1) * spread_return
    
    return spread, z_score, strategy_returns.fillna(0)

# 1. Setup Periods
IS_START, IS_END = "2013-01-01", "2014-12-31"
OOS_START, OOS_END = "2015-01-01", "2022-12-31"
# Fetch extra buffer for the rolling window z-score
FULL_DATA = fetch_log_prices(['EWA', 'EWC'], IS_START, OOS_END)

# 2. Training (In-Sample)
is_data = FULL_DATA.loc[IS_START:IS_END]
X_is = sm.add_constant(is_data['EWC'])
model = sm.OLS(is_data['EWA'], X_is).fit()
gamma_fixed = model.params['EWC']
intercept_fixed = model.params['const']

# Optimization: Find best threshold on IS data
best_sharpe = -np.inf
best_threshold = 1.0
for delta in np.linspace(0.5, 2.0, 16):
    _, _, ret = backtest_strategy(is_data, gamma_fixed, intercept_fixed, delta)
    sharpe = np.sqrt(252) * ret.mean() / ret.std() if ret.std() != 0 else 0
    if sharpe > best_sharpe:
        best_sharpe = sharpe
        best_threshold = delta

print(f"Optimized Threshold (IS): {best_threshold:.2f}")

# 3. Build full-period series (IS + OOS)
# Run on FULL_DATA so rolling stats are computed with the pre-IS buffer.
spread_all, z_score_all, returns_all = backtest_strategy(
    FULL_DATA, gamma_fixed, intercept_fixed, best_threshold
)

# Plot window includes both IS and OOS
plot_start, plot_end = IS_START, OOS_END
spread_plot = spread_all.loc[plot_start:plot_end]
z_plot = z_score_all.loc[plot_start:plot_end]

# Cumulative return: force IS period to 0, accumulate only in OOS
plot_index = FULL_DATA.loc[plot_start:plot_end].index
cum_returns = pd.Series(0.0, index=plot_index)

oos_returns = returns_all.loc[OOS_START:OOS_END]
cum_returns.loc[OOS_START:OOS_END] = oos_returns.cumsum()

# 4. Plotting (IS + OOS)
fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

ax1.plot(spread_plot, color='black', lw=1)
ax1.axvline(pd.Timestamp(OOS_START), color='gray', linestyle='--', lw=1)
ax1.set_title("Spread (IS + OOS)")
ax1.grid(True, alpha=0.3)

ax2.plot(z_plot, color='blue', lw=1)
ax2.axhline(best_threshold, color='red', linestyle='--')
ax2.axhline(-best_threshold, color='red', linestyle='--')
ax2.axhline(0, color='black', lw=0.5)
ax2.axvline(pd.Timestamp(OOS_START), color='gray', linestyle='--', lw=1)
ax2.set_title(f"Z-Score (IS + OOS, Threshold $\\Delta$={best_threshold:.2f})")
ax2.grid(True, alpha=0.3)

ax3.plot(cum_returns, color='green', lw=2)
ax3.axvline(pd.Timestamp(OOS_START), color='gray', linestyle='--', lw=1)
ax3.set_title("Cumulative Return (IS fixed at 0, OOS accumulated)")
ax3.set_ylabel("Log Profit")
ax3.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()